Helical Propensity Calculation From Simulation Trajectory

In [ ]:
import gsd.hoomd
import numpy as np
import freud
import math
import matplotlib.pyplot as plt
from matplotlib import rcParams
import os
import pandas as pd

def analyzedata_sets(sets):
    rcParams['font.family'] = 'serif'
    rcParams['font.size'] = 12

    bias_values = np.linspace(0.75, 1.0, 12)

    for dataset in sets:
        set_name = dataset["name"]
        folder_path = dataset["base_path"]
        output_folder = os.path.join(set_name)
        os.makedirs(output_folder, exist_ok=True)

        print(f"\n📂 Processing dataset: {set_name}")
        avg_helicity_per_bias = []

        for bias in bias_values:
            # Handle filenames
            if bias in [0.75, 1.0]:
                trajectory_name = os.path.join(folder_path, f"trajectory_bias_{bias}.gsd")
            else:
                trajectory_name = os.path.join(folder_path, f"trajectory_bias_{bias:.16f}.gsd")

            if not os.path.isfile(trajectory_name):
                print(f"⚠️  File {trajectory_name} not found. Skipping.")
                continue

            print(f"  Processing bias {bias:.2f} -> {trajectory_name}")
            bcd = gsd.hoomd.open(trajectory_name, 'r')
            traj = bcd[10000:20000]
            frame = traj[0]

            N_particles = frame.particles.N
            N_chains = 64
            particles_per_chain = N_particles // N_chains

            # Build dihedral tables
            tables_psi = [dict() for _ in range(N_chains)]
            tables_phi = [dict() for _ in range(N_chains)]

            dihedral_types = frame.dihedrals.types
            dihedrals_psi = frame.dihedrals.typeid == dihedral_types.index('psi')
            dihedrals_phi = frame.dihedrals.typeid == dihedral_types.index('phi')

            for idx, group in enumerate(frame.dihedrals.group):
                chain_idx = group[0] // particles_per_chain
                if dihedrals_psi[idx]:
                    tables_psi[chain_idx][group[1]] = group
                if dihedrals_phi[idx]:
                    tables_phi[chain_idx][group[2]] = group

            # Remove mismatches
            for chain in range(N_chains):
                mismatched_keys = tables_phi[chain].keys() ^ tables_psi[chain].keys()
                for key in mismatched_keys:
                    tables_psi[chain].pop(key, None)
                    tables_phi[chain].pop(key, None)

            # Store dihedrals
            all_timeseries = [{k: np.zeros((len(traj), 2)) for k in tables_psi[chain].keys()} 
                              for chain in range(N_chains)]
            gyration_time = np.zeros((len(traj), N_chains))

            def compute_dihedral(array):
                b1 = array[1, :] - array[0, :]
                b2 = array[2, :] - array[1, :]
                b3 = array[3, :] - array[2, :]
                b2 /= np.linalg.norm(b2)
                n1 = np.cross(b1, b2); n1 /= np.linalg.norm(n1)
                n2 = np.cross(b2, b3); n2 /= np.linalg.norm(n2)
                m1 = np.cross(n1, b2)
                x = np.dot(n1, n2); y = np.dot(m1, n2)
                return -np.arctan2(y, x)

            for time, frame in enumerate(traj):
                freud_box = freud.box.Box.from_box(frame.configuration.box)
                unwrapped = freud_box.unwrap(frame.particles.position, frame.particles.image)

                for chain in range(N_chains):
                    start = chain * particles_per_chain
                    end = (chain + 1) * particles_per_chain
                    chain_pos = frame.particles.position[start:end]
                    chain_mass = frame.particles.mass[start:end]
                    com = np.sum(chain_mass[:, None] * chain_pos, axis=0) / np.sum(chain_mass)
                    gyration_time[time, chain] = np.sqrt(np.sum(chain_mass * np.sum((chain_pos - com)**2, axis=1)) / np.sum(chain_mass))

                    for key in tables_psi[chain].keys():
                        psi_g = tables_psi[chain][key]
                        phi_g = tables_phi[chain][key]
                        psi = compute_dihedral(unwrapped[psi_g, :])
                        phi = compute_dihedral(unwrapped[phi_g, :])
                        all_timeseries[chain][key][time] = np.array([phi, psi])

            # Helicity detection
            def is_helical_t(angles):
                helix_phi = (angles[:, 0] > math.radians(-160)) & (angles[:, 0] < math.radians(-20))
                helix_psi = (angles[:, 1] > math.radians(-120)) & (angles[:, 1] < math.radians(50))
                return helix_phi & helix_psi

            all_timeseries_helix = []
            for chain in range(N_chains):
                timeseries = all_timeseries[chain]
                all_keys = list(timeseries.keys())
                valid_helix_keys = all_keys[1:-1]
                timeseries_helix = {}
                for res, key in enumerate(valid_helix_keys):
                    left = all_keys[all_keys.index(key) - 1]
                    right = all_keys[all_keys.index(key) + 1]
                    my_angles_h = is_helical_t(timeseries[key])
                    my_left_angles_h = is_helical_t(timeseries[left])
                    my_right_angles_h = is_helical_t(timeseries[right])
                    is_helix = my_angles_h & my_left_angles_h & my_right_angles_h
                    timeseries_helix[res+2] = is_helix
                all_timeseries_helix.append(timeseries_helix)

            # Propensity calculation
            num_residues = len(all_timeseries_helix[0])
            residues = list(range(2, num_residues + 2))
            chain_propensities = np.zeros((N_chains, num_residues))
            for chain in range(N_chains):
                for residue, helicities in all_timeseries_helix[chain].items():
                    res_idx = residue - 2
                    chain_propensities[chain, res_idx] = np.mean(helicities)

            avg_propensities = np.mean(chain_propensities, axis=0)
            std_propensities = np.std(chain_propensities, axis=0)

            # Save per-residue CSV
            df_prop = pd.DataFrame({
                "Residue": residues,
                "Average Propensity": avg_propensities,
                "Standard Deviation": std_propensities
            })
            df_prop.to_csv(os.path.join(output_folder, f'propensities_{bias:.2f}.csv'), index=False)

            # Plot per-residue helicity
            plt.figure(figsize=(12, 6))
            plt.errorbar(residues, avg_propensities, yerr=std_propensities, fmt='o-', markersize=5,
                         capsize=3, color='green', ecolor='lightgray', elinewidth=2)
            plt.xlabel('Residue Number')
            plt.ylabel('Helicity Propensity')
            plt.title(f'{set_name}: Helicity Propensity (Bias {bias:.2f})')
            plt.grid(True, linestyle='--', alpha=0.6)
            plt.tight_layout()
            plt.savefig(os.path.join(output_folder, f'propensities_{bias:.2f}.png'), dpi=300)
            plt.close()

            # Chain helicity with per-chain STD
            chain_helicity_mean = np.mean(chain_propensities, axis=1)
            chain_helicity_std = np.std(chain_propensities, axis=1)
            df_chain = pd.DataFrame({
                "Chain": np.arange(N_chains),
                "Average_Helicity": chain_helicity_mean,
                "STD_Helicity": chain_helicity_std
            })
            df_chain.to_csv(os.path.join(output_folder, f"chain_helicity_bias_{bias:.2f}.csv"), index=False)

            # Bar plot
            plt.figure(figsize=(14, 6))
            x_pos = np.arange(N_chains)
            plt.bar(x_pos, chain_helicity_mean, yerr=chain_helicity_std,
                    color='skyblue', error_kw=dict(elinewidth=1, ecolor='red', capsize=3))
            plt.axhline(y=np.mean(chain_helicity_mean), color='red', linestyle='--', label=f'Average: {np.mean(chain_helicity_mean):.3f}')
            plt.xlabel('Chain Index')
            plt.ylabel('Average Helicity Propensity')
            plt.title(f'{set_name}: Helicity Across Chains (Bias {bias:.2f})')
            plt.xticks(x_pos[::5])
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.6, axis='y')
            plt.tight_layout()
            plt.savefig(os.path.join(output_folder, f'chain_helicity_{bias:.2f}.png'), dpi=300)
            plt.close()

            avg_helicity_per_bias.append([bias, np.mean(chain_helicity_mean), np.std(chain_helicity_mean)])

        # Final plot: avg helicity vs bias
        df_avg = pd.DataFrame(avg_helicity_per_bias, columns=["Bias", "Average_Helicity", "STD_Helicity"])
        df_avg.to_csv(os.path.join(output_folder, "average_helicity_vs_bias.csv"), index=False)

        plt.figure(figsize=(10, 6))
        plt.errorbar(df_avg["Bias"], df_avg["Average_Helicity"], yerr=df_avg["STD_Helicity"],
                     marker='o', capsize=4, linewidth=2)
        plt.xlabel("Bias")
        plt.ylabel("Average Helicity Propensity")
        plt.title(f"{set_name}: Average Helicity vs Bias")
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.savefig(os.path.join(output_folder, "average_helicity_vs_bias.png"), dpi=300)
        plt.close()
        print(f" Done with dataset '{set_name}'. Results saved in folder '{output_folder}'.")

# === Define sets ===
sets = [
    {"name": "Full_length", "base_path": "/home/POLY/bhandarit/Desktop/mount_viper/chain_64_01/workspace/43ef74dd463bfa6f52acb921ee73e162/"},
    {"name": "H0_H3", "base_path": "/home/POLY/bhandarit/Desktop/mount_viper/chain_64_01/workspace/3de12d848c06a70b5668c64369550562/"},
    {"name": "H4_H6", "base_path": "/home/POLY/bhandarit/Desktop/mount_viper/chain_64_01/workspace/7fd5f713b9bd4167bafd3d2d9378bc51/"}
]

# Run the analysis
analyzedata_sets(sets)
